# **3. VAR use in match reports**

This study searches for textual evidence of VAR in the **Reason for delayed start and/or restart, and added time** field and in the narrative body of the match reports. The search covers checks/reviews, VAR/VOR/AVAR, video refereeing, monitor, and ARA (referee review area), while ignoring the fixed officiating-team table and purely financial records.

The unit of analysis is the **match**. Because a match report usually records only that a check or review occurred—without stating how many incidents were analyzed—the rates below represent matches with at least one mention, not the actual number of VAR interventions. The result is associated with both participating teams and the on-field referee solely as a characteristic of that match; it neither attributes VAR activity to one team nor assesses which team benefited from a decision.

## **1. Extracting the match-report field**

In [1]:
from __future__ import annotations

import hashlib
import json
import re
import shutil
import subprocess
import unicodedata
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd
import pdfplumber
from tqdm.auto import tqdm

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
INDEX_PATH = ROOT / "data" / "raw" / "match-reports" / "match_reports_index.csv"
CACHE_PATH = ROOT / "data" / "interim" / "match_report_notes.csv"
CACHE_META_PATH = ROOT / "data" / "interim" / "match_report_notes.json"
OUTPUT_DIR = ROOT / "data" / "processed" / "var_usage"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

index = pd.read_csv(INDEX_PATH)
index = index.loc[index["pdf_path"].notna()].copy()
index["pdf_file"] = index["pdf_path"].map(lambda value: ROOT / value)
index = index.loc[index["pdf_file"].map(Path.exists)].copy()
print(f"{len(index):,} local match reports available for analysis")

2,604 local match reports available for analysis


In [2]:
SECTION_START = re.compile(
    r"Motivo\s+de\s+atraso\s+no\s+in[ií]cio\s+e/ou\s+rein[ií]cio,?\s+e\s+de\s+acr[eé]scimos\s*:",
    re.IGNORECASE,
)
SECTION_END = re.compile(r"\n\s*Observa[cç][oõ]es\s+Eventuais\s*\n", re.IGNORECASE)
OCCURRENCES_START = re.compile(r"\n\s*Ocorr[eê]ncias\s*/\s*Observa[cç][oõ]es\s*\n", re.IGNORECASE)


def normalize(value: object) -> str:
    text = unicodedata.normalize("NFKD", str(value))
    text = "".join(character for character in text if not unicodedata.combining(character))
    return re.sub(r"\s+", " ", text.casefold()).strip()


def extract_pdf_text(path: Path) -> str:
    if shutil.which("pdftotext"):
        result = subprocess.run(
            ["pdftotext", "-layout", str(path), "-"],
            capture_output=True, text=True, errors="replace", check=True,
        )
        return result.stdout
    with pdfplumber.open(path) as pdf:
        return "\n".join((page.extract_text() or "") for page in pdf.pages)


def extract_delay_reason(text: str) -> str | None:
    start = SECTION_START.search(text)
    if not start:
        return None
    tail = text[start.end():]
    end = SECTION_END.search(tail)
    value = tail[:end.start()] if end else tail[:3000]
    value = re.sub(r"\s+", " ", value).strip()
    if value:
        return value

    prefix = text[:start.start()]
    occurrence_starts = list(OCCURRENCES_START.finditer(prefix))
    if occurrence_starts:
        fallback = re.sub(r"\s+", " ", prefix[occurrence_starts[-1].end():]).strip()
        fallback_normalized = normalize(fallback)
        system_failure = (
            "problemas no sistema" in fallback_normalized
            or "erro de sistema" in fallback_normalized
        )
        if system_failure and ("motivo de atraso" in fallback_normalized or "parte do scout" in fallback_normalized):
            return fallback
    return None


def parse_report(row) -> dict:
    text = extract_pdf_text(Path(row.pdf_file))
    return {
        "game_id": int(row.game_id),
        "delay_reason": extract_delay_reason(text),
        "report_text_normalized": normalize(text),
    }

In [3]:
signature_source = "|".join(
    f"{row.game_id}:{Path(row.pdf_file).stat().st_size}" for row in index.itertuples(index=False)
)
parser_signature = hashlib.sha256(("var-notes-v2|" + signature_source).encode()).hexdigest()
cached_signature = None
if CACHE_META_PATH.exists():
    cached_signature = json.loads(CACHE_META_PATH.read_text(encoding="utf-8")).get("parser_signature")

if CACHE_PATH.exists() and cached_signature == parser_signature:
    notes = pd.read_csv(CACHE_PATH)
else:
    workers = min(10, max(1, len(index))) if shutil.which("pdftotext") else 1
    records = []
    with ThreadPoolExecutor(max_workers=workers) as executor:
        iterator = executor.map(parse_report, index.itertuples(index=False))
        records = list(tqdm(iterator, total=len(index), desc="Reading match reports"))
    notes = pd.DataFrame(records)
    notes.to_csv(CACHE_PATH, index=False)
    CACHE_META_PATH.write_text(
        json.dumps({"parser_signature": parser_signature}, indent=2), encoding="utf-8"
    )

print(f"Field found in {notes.delay_reason.notna().sum():,} of {len(notes):,} match reports")

Reading match reports:   0%|          | 0/2604 [00:00<?, ?it/s]

Field found in 2,602 of 2,604 match reports


## **2. Identification rules**

Classification is binary by match: if VAR is mentioned in the delays/added-time field or in the narrative body of the report, the match is considered to have used VAR. The administrative officiating table is removed before searching the body because it lists the video team even when no use is reported. This measures occurrence by match, not the actual number of interventions.

In [4]:
VAR_PATTERN = re.compile(
    r"(?:"
    r"\b(?:var|vor)(?=\b|foi\b)|\bavar\d*\b"
    r"|\b(?:arbitro|arbitra)\s+(?:assistente\s+)?de\s+video\b"
    r"|\barbitragem\s+de\s+video\b"
    r"|\bvideo\s+assistant\s+referee\b"
    r")"
)
ARA_PATTERN = re.compile(
    r"(?:"
    r"\ba\s*[.]\s*r\s*[.]\s*a\s*[.]?"
    r"|\barea\s+(?:de\s+)?revisao(?:\s+do\s+arbitro)?\b"
    r"|\b(?:na|no|da|do|a|ao|ate\s+a|para\s+a)\s+(?:a\s+)?ara(?=\b|foi\b)"
    r"|\b(?:chec\w*|revis\w*|dirig\w*|ida)\b.{0,50}\bara(?=\b|foi\b)"
    r")"
)
DELAY_ACTION_PATTERN = re.compile(
    r"(?:\b(?:chec\w*|revis\w*)\b|\banalis\w*\b.{0,40}\bvideo\b|\btempo\s+de\s+cabine\b)"
)
VIDEO_ACTION_PATTERN = re.compile(
    r"(?:\b(?:chec\w*|revis\w*|analis\w*)\b.{0,50}\bvideo\b"
    r"|\bvideo\b.{0,50}\b(?:chec\w*|revis\w*|analis\w*)\b)"
)
MONITOR_PATTERN = re.compile(
    r"(?:"
    r"\b(?:fui|me\s+dirigi|dirigi-me|me\s+desloquei|desloquei-me|me\s+encaminhei|encaminhei-me)\b.{0,80}"
    r"\b(?:ao|ate|para|a)\s+(?:o\s+)?(?:monitor|ara|area\s+(?:de\s+)?revisao)\b"
    r"|\b(?:minha\s+ida|eu\s+me\s+dirigia|caminhada\s+em\s+direcao)\b.{0,140}\b(?:monitor|area\s+(?:de\s+)?revisao|ara)\b"
    r"|\bapos\s+(?:olhar|analisar|a\s+analise|analise|visualizar|revisar)\w*\b.{0,120}\b(?:monitor|area\s+(?:de\s+)?revisao|ara)\b"
    r"|\bapos\s+(?:revisao|visualizacao)\b.{0,80}\b(?:monitor|area\s+(?:de\s+)?revisao|ara)\b"
    r"|\brevisao\s+na\s+[\"']?ara\b"
    r")"
)
ADMIN_VAR_ROSTER_PATTERN = re.compile(r"\barbitragem\b.*?\bcronologia\b", re.DOTALL)
ADMIN_PAYMENT_PATTERN = re.compile(
    r"(?:"
    r"\b(?:o\s+pagamento\s+da\s+taxa|as\s+taxas\s+referentes|a\s+taxa\s+de\s+arbitragem)\b"
    r"|\barbitro\s+de\s+video:\s*agencia\b"
    r").*?(?=\b(?:motivo\s+de\s+atraso|observacoes\s+eventuais|relatorio\s+do\s+assistente|substituicoes)\b)",
    re.DOTALL,
)
DELAY_VAR_PATTERN = re.compile(
    "|".join(f"(?:{pattern.pattern})" for pattern in (VAR_PATTERN, ARA_PATTERN, DELAY_ACTION_PATTERN))
)
BODY_VAR_PATTERN = re.compile(
    "|".join(f"(?:{pattern.pattern})" for pattern in (VAR_PATTERN, ARA_PATTERN, VIDEO_ACTION_PATTERN, MONITOR_PATTERN))
)


def evidence_excerpt(text: str, pattern: re.Pattern, radius: int = 130) -> str | None:
    match = pattern.search(text)
    if not match:
        return None
    start = max(0, match.start() - radius)
    end = min(len(text), match.end() + radius)
    return text[start:end].strip()


def narrative_body(text: str) -> str:
    without_roster = ADMIN_VAR_ROSTER_PATTERN.sub(" ", text, count=1)
    return ADMIN_PAYMENT_PATTERN.sub(" ", without_roster)


notes["delay_reason_normalized"] = notes["delay_reason"].fillna("").map(normalize)
notes["report_body_normalized"] = notes["report_text_normalized"].map(narrative_body)
notes["section_mentions_var"] = notes["delay_reason_normalized"].map(lambda value: bool(DELAY_VAR_PATTERN.search(value)))
notes["body_mentions_var"] = notes["report_body_normalized"].map(lambda value: bool(BODY_VAR_PATTERN.search(value)))
notes["went_to_monitor"] = notes["report_body_normalized"].map(lambda value: bool(MONITOR_PATTERN.search(value)))
notes["had_var"] = notes["section_mentions_var"] | notes["body_mentions_var"]
notes["var_evidence_source"] = [
    "acrescimos" if section else ("corpo" if body else None)
    for section, body in zip(notes["section_mentions_var"], notes["body_mentions_var"])
]
notes["var_evidence"] = [
    evidence_excerpt(delay, DELAY_VAR_PATTERN) if section else evidence_excerpt(body, BODY_VAR_PATTERN)
    for delay, body, section in zip(
        notes["delay_reason_normalized"], notes["report_body_normalized"], notes["section_mentions_var"]
    )
]

display(notes[["had_var"]].sum().rename("partidas").to_frame())

,partidas
had_var,1147


### Rule sanity check

In [5]:
delay_cases = [
    ("acréscimos por checagens do VAR", True),
    ("tempo de checagem da cabine VOR e substituições", True),
    ("acréscimos devido à revisão do árbitro de vídeo", True),
    ("acréscimos devido a revisões na A.R.A.", True),
    ("acréscimos devido a checagens e substituições", True),
    ("acréscimos por análise de vídeo", True),
    ("substituição do árbitro pelo AVAR1", True),
    ("atendimento médico e substituições", False),
]
for text, expected_var in delay_cases:
    value = normalize(text)
    assert bool(DELAY_VAR_PATTERN.search(value)) == expected_var

monitor_cases = [
    "após revisão sugerida pelo VAR, fui ao monitor e alterei a decisão",
    "após olhar no monitor à beira do campo, mantive a decisão",
    "após revisar na ARA, apliquei cartão vermelho",
]
assert all(MONITOR_PATTERN.search(normalize(text)) for text in monitor_cases)
assert not MONITOR_PATTERN.search(normalize("o atleta chutou a cabine e o monitor do VAR"))
assert not ARA_PATTERN.search(normalize("jose roberto assuncao de araujo"))
assert not BODY_VAR_PATTERN.search(narrative_body(normalize("arbitragem var: fulano cronologia nada houve")))
assert not BODY_VAR_PATTERN.search(narrative_body(normalize("observacoes: o pagamento da taxa de arbitragem foi efetuado; var: cheque 123 motivo de atraso: nada houve")))
print(f"{len(delay_cases) + len(monitor_cases) + 4} sanity checks passed")

15 sanity checks passed


## **3. Profiles by season, team, and referee**

In [6]:
TEAM_ALIASES = {
    "America": "América-MG", "América": "América-MG",
    "Athletico Paranaense": "Athletico-PR",
    "Atlético": "Atlético-GO", "Atlético Mineiro": "Atlético-MG",
    "Atlético Goianiense Saf": "Atlético-GO",
    "Botafogo de Futebol E Regatas": "Botafogo",
    "Coritiba SAF": "Coritiba",
    "Esporte Clube Bahia": "Bahia",
    "Fortaleza Esporte Clube": "Fortaleza", "Fortaleza SAF": "Fortaleza",
    "Santos FC": "Santos", "Sport Recife": "Sport",
    "Vasco da Gama": "Vasco", "Vasco da Gama Saf": "Vasco",
}

metadata_columns = [
    "game_id", "match_date", "year", "competition", "category", "round",
    "home_team", "away_team", "referee",
]
matches = index[metadata_columns].merge(notes.drop(columns=["report_text_normalized", "report_body_normalized"]), on="game_id", how="left", validate="one_to_one")
matches["home_team"] = matches["home_team"].replace(TEAM_ALIASES)
matches["away_team"] = matches["away_team"].replace(TEAM_ALIASES)


def build_profile(frame: pd.DataFrame, dimension: str) -> pd.DataFrame:
    grouped = frame.groupby(dimension, dropna=False)
    profile = grouped.agg(
        games=("game_id", "nunique"),
        var_games=("had_var", "sum"),
    ).reset_index().rename(columns={dimension: "entity"})
    profile["var_per_game"] = profile["var_games"] / profile["games"]
    return profile


team_matches = pd.concat(
    [
        matches.rename(columns={"home_team": "team"}),
        matches.rename(columns={"away_team": "team"}),
    ],
    ignore_index=True,
)
season_profile = build_profile(matches, "year")
team_profile = build_profile(team_matches, "team")
referee_profile = build_profile(matches.loc[matches.referee.notna()], "referee")

display(season_profile)
display(team_profile.sort_values("var_per_game", ascending=False).head(10))
display(referee_profile.loc[referee_profile.games > 10].sort_values("var_per_game", ascending=False).head(10))

,entity,games,var_games,var_per_game
0,2020,380,206,0.542105
1,2021,380,171,0.450000
2,2022,380,178,0.468421
3,2023,380,165,0.434211
4,2024,380,156,0.410526
5,2025,380,161,0.423684
6,2026,324,110,0.339506


,entity,games,var_games,var_per_game
17,Goiás,114,62,0.543860
27,São Paulo,261,138,0.528736
26,Sport,114,56,0.491228
21,Mirassol,70,34,0.485714
22,Palmeiras,261,123,0.471264
16,Fortaleza,228,107,0.469298
6,Botafogo,223,104,0.466368
2,Atlético-GO,152,70,0.460526
5,Bahia,223,102,0.457399
23,Red Bull Bragantino,261,117,0.448276


,entity,games,var_games,var_per_game
11,Braulio da Silva Machado,120,73,0.608333
62,Ricardo Marques Ribeiro,24,14,0.583333
48,Luiz Flavio de Oliveira,67,39,0.582090
65,Rodrigo D Alonso Ferreira,14,8,0.571429
63,Rodolpho Toski Marques,30,17,0.566667
57,Paulo Roberto Alves Junior,39,22,0.564103
37,Jefferson Ferreira de Moraes,35,19,0.542857
77,Yuri Elino Ferreira da Cruz,13,7,0.538462
29,Fernando Antonio Mendes de Salles Nascimento F...,17,9,0.529412
59,Rafael Traci,19,10,0.526316


## **4. Export and audit**

In [7]:
matches.to_csv(OUTPUT_DIR / "match_var_usage.csv", index=False)
season_profile.to_csv(OUTPUT_DIR / "var_profile_by_season.csv", index=False)
team_profile.to_csv(OUTPUT_DIR / "var_profile_by_team.csv", index=False)
referee_profile.to_csv(OUTPUT_DIR / "var_profile_by_referee.csv", index=False)

latest_date = pd.to_datetime(matches.match_date.str.strip(), dayfirst=True, errors="coerce").max()
manifest = {
    "games": int(len(matches)),
    "latest_match_date": latest_date.strftime("%Y-%m-%d"),
    "section_found": int(matches.delay_reason.notna().sum()),
    "var_games": int(matches["had_var"].sum()),
    "unit": "partidas com ao menos uma menção",
}
(OUTPUT_DIR / "run_manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)

def website_rows(profile: pd.DataFrame) -> list[dict]:
    columns = ["entity", "games", "var_games"]
    rows = profile[columns].copy()
    rows["entity"] = rows["entity"].astype(str)
    return rows.to_dict(orient="records")

site_data = {
    "manifest": manifest,
    "by_year": website_rows(season_profile),
    "by_team": website_rows(team_profile),
    "by_referee": website_rows(referee_profile.loc[referee_profile.games > 10]),
    "matches_with_var": matches.loc[matches.had_var, "game_id"].astype(str).tolist(),
}
(OUTPUT_DIR / "site_var_data.json").write_text(
    json.dumps(site_data, ensure_ascii=False, separators=(",", ":")), encoding="utf-8"
)

audit = matches.loc[matches.had_var, [
    "game_id", "year", "home_team", "away_team", "referee",
    "delay_reason", "var_evidence_source", "var_evidence",
]].sort_values(["year", "game_id"])
audit.to_csv(OUTPUT_DIR / "var_mentions_audit.csv", index=False)
display(pd.Series(manifest, name="value").to_frame())
display(audit.tail(10))

,value
games,2604
latest_match_date,2026-10-28
section_found,2602
var_games,1147
unit,partidas com ao menos uma menção


,game_id,year,home_team,away_team,referee,delay_reason,var_evidence_source,var_evidence
2584,832196,2026,Atlético-MG,Coritiba,Davi De Oliveira Lacerda,Acréscimos devido atendimento médico em atleta...,acrescimos,acrescimos devido atendimento medico em atleta...
2585,832197,2026,Grêmio,Cruzeiro,Joao Vitor Gobi,"Acréscimos devido as substituições, atendiment...",acrescimos,"acrescimos devido as substituicoes, atendiment..."
2587,832199,2026,Bahia,Flamengo,Wilton Pereira Sampaio,A partida teve um atraso de 4 minutos no seu r...,acrescimos,"ar o referido atraso, porem nao foi possivel. ..."
2588,832200,2026,Remo,Red Bull Bragantino,Alex Gomes Stefano,Acréscimos devido a parada médica para resfria...,acrescimos,"mos devido a parada medica para resfriamento, ..."
2591,832203,2026,Santos,Bahia,Savio Pereira Sampaio,Houve atraso de 10 minutos no reinicio da part...,acrescimos,da partida devido a procedimentos realizado po...
2592,832204,2026,Palmeiras,Red Bull Bragantino,Braulio da Silva Machado,"Acréscimos motivados por substituições, avalia...",acrescimos,"acrescimos motivados por substituicoes, avalia..."
2593,832205,2026,Mirassol,São Paulo,Lucas Casagrande,A partida teve um atraso de 1 minuto no seu re...,acrescimos,"itar o referido atraso, porem nao foi possivel..."
2595,832207,2026,Internacional,Botafogo,Davi De Oliveira Lacerda,Informo que as duas equipes retornaram do inte...,acrescimos,"(tres) minutos, sendo assim a partida teve o s..."
2598,832210,2026,Chapecoense,Fluminense,Anderson Daronco,Informo que houve atraso de 2 minutos no iníci...,acrescimos,o de radio intercomunicador da equipe de arbit...
2599,832211,2026,Fluminense,Internacional,Lucas Casagrande,"Tempo acrescido devido as substituições, atend...",acrescimos,"tempo acrescido devido as substituicoes, atend..."
